# 01 — AequilibraE projects and networks

**AequilibraE** is a fully-featured, open-source transportation modeling package for Python.
This notebook series walks through a complete transport-modeling workflow, using
**lonboard** for interactive, fully offline WebGL maps inside JupyterLab.

In this first notebook we:

1. create a model from a bundled example (Coquimbo/La Serena, Chile);
2. look at how an AequilibraE *project* is structured (a SQLite/SpatiaLite database);
3. load the network's links, nodes and zones as GeoDataFrames;
4. put everything on an interactive map.

> **Note on GIS files** — an AequilibraE project is a standard *SpatiaLite* database.
> This fork reads and writes that format with a pure-Python engine
> (shapely + pyproj + SQLite's built-in R\*Tree index), so installing the fork's wheel
> is all you need — no `mod_spatialite` system package, on any OS. The files remain
> 100% compatible with QGIS and any other SpatiaLite-aware tool.

**Requirements**: this fork of AequilibraE plus `jupytergis` — see the [setup instructions](README.md). Do **not** `pip install aequilibrae` from PyPI: that installs the upstream package, which still needs native SpatiaLite. Run inside JupyterLab to see the maps.

In [1]:
from pathlib import Path
from tempfile import gettempdir
from uuid import uuid4

from aequilibrae.utils.create_example import create_example

# Every notebook in this series works inside a throw-away folder
fldr = str(Path(gettempdir()) / uuid4().hex)

# 'coquimbo' is a real-world model of Coquimbo/La Serena, Chile.
# Other options: 'sioux_falls' (the classic toy network) and 'nauru'.
project = create_example(fldr, "coquimbo")
project

## What is inside a project?

The project folder holds a handful of files — the two SQLite databases are the model itself:

- `project_database.sqlite` — network (links, nodes), zones, matrix index, results index…
- `public_transport.sqlite` — GTFS-derived transit network (created on demand);
- `matrices/` — demand and skim matrices (OMX / AEM files);
- `parameters.yml` — model parameters.

Because it is *just SQLite*, you can open it with any SQL client — or QGIS.


In [2]:
import pandas as pd

with project.db_connection as conn:
    tables = pd.read_sql("SELECT name, type FROM sqlite_master WHERE type IN ('table','view') ORDER BY name", conn)
tables[~tables.name.str.startswith(("idx_", "sqlite_", "geometry_", "spatial_", "views_", "virts_"))].head(20)

,name,type
0,ElementaryGeometries,table
1,KNN,table
2,SpatialIndex,table
3,about,table
4,attributes_documentation,table
5,data_licenses,table
6,geom_cols_ref_sys,view
24,link_types,table
25,links,table
26,matrices,table


## The network

`project.network` gives access to links and nodes. The `.data` accessors return
**GeoDataFrames** (WGS84 / EPSG:4326), which makes the whole geopandas/shapely
ecosystem available directly.


In [3]:
links = project.network.links.data
nodes = project.network.nodes.data
zones = project.zoning.data

print(f"{len(links)} links, {len(nodes)} nodes, {len(zones)} zones")
links[["link_id", "a_node", "b_node", "direction", "distance", "modes", "link_type"]].head()

19983 links, 15724 nodes, 133 zones


,link_id,a_node,b_node,direction,distance,modes,link_type
0,1,64158,64194,0,15.192014,ct,residential
1,2,64208,64194,1,117.170146,ct,residential
2,3,47501,47539,1,37.343065,ct,residential
3,12,78052,78051,1,47.825940,ct,residential
4,13,73608,79808,1,93.536834,ct,residential


In [4]:
# Modes available in this model, straight from SQL
with project.db_connection as conn:
    modes = pd.read_sql("SELECT mode_name, mode_id, description FROM modes", conn)
modes

,mode_name,mode_id,description
0,car,c,All motorized vehicles
1,transit,t,Public transport vehicles
2,walk,w,Walking links
3,bicycle,b,Biking links


## Interactive map with JupyterGIS

`GISDocument` builds a collaborative map document rendered by JupyterLab.
Every layer you add appears in the layer tree on the left of the map widget,
where symbology can also be edited interactively.


In [5]:
# Offline map helper ---------------------------------------------------------
# Interactive maps with no server extensions, no labextensions beyond the
# ipywidgets manager, and no CDN: lonboard renders WebGL maps whose frontend
# JavaScript ships from the kernel through the ipywidgets channel.
#
# Backends (AEQ_MAP_BACKEND environment variable):
#   lonboard (default) - interactive WebGL maps (pip install lonboard anywidget)
#   static             - matplotlib rendering, works absolutely anywhere
#
# The declarative symbology below (field()/constant() chains) is self-contained
# and renders identically on both backends.
import os

import matplotlib.colors
import matplotlib.pyplot as _plt
import numpy as np


# --- declarative symbology --------------------------------------------------
class _Mapping:
    def __init__(self, field, scheme, params):
        self.field, self.scheme, self.params = field, scheme, params

    def encoding(self, *targets):
        return {"field": self.field, "scheme": self.scheme,
                "params": self.params, "encodings": list(targets)}


class _Field:
    def __init__(self, name):
        self.name = name

    def colormap(self, name="viridis", *, domain=None, reverse=False, n_shades=9):
        return _Mapping(self.name, "colormap",
                        {"name": name, "domain": domain, "reverse": reverse})

    def scalar(self, *, domain, output_range):
        return _Mapping(self.name, "scalar",
                        {"domain": list(domain), "range": list(output_range)})

    def categorical(self, name="tab10"):
        return _Mapping(self.name, "categorical", {"name": name})


class _Constant:
    def __init__(self, value):
        self.value = value

    def encoding(self, *targets):
        scheme = "constant_num" if isinstance(self.value, (int, float)) else "constant_color"
        return {"field": None, "scheme": scheme,
                "params": {"value": self.value}, "encodings": list(targets)}


def field(name):
    """Style by a data column: .colormap() / .scalar() / .categorical()."""
    return _Field(name)


def constant(value):
    """A fixed colour (hex/name) or number, e.g. constant("#dc2626")."""
    return _Constant(value)


def _rgba255(c, alpha=1.0):
    r, g, b, a = matplotlib.colors.to_rgba(c, alpha)
    return [int(r * 255), int(g * 255), int(b * 255), int(a * 255)]


def _style_arrays(symbology, gdf):
    """symbology -> per-row uint8 RGBA arrays and float width arrays."""
    n = len(gdf)
    out = {"stroke": None, "width": None, "fill": None}
    if not symbology:
        return out
    mappings = [m for group in symbology for m in (group if isinstance(group, list) else [group])]
    for m in mappings:
        scheme, params, fld, encs = m["scheme"], m["params"], m["field"], m["encodings"]
        arr = wid = None
        if scheme == "constant_color":
            arr = np.tile(_rgba255(params["value"]), (n, 1)).astype(np.uint8)
        elif scheme == "colormap":
            cmap = _plt.get_cmap(params["name"])
            if params.get("reverse"):
                cmap = cmap.reversed()
            dom = params.get("domain") or [float(gdf[fld].min()), float(gdf[fld].max())]
            vals = gdf[fld].to_numpy(dtype=float)
            t = np.clip((vals - dom[0]) / max(dom[1] - dom[0], 1e-12), 0, 1)
            rgba = cmap(t)
            arr = (rgba * 255).astype(np.uint8)
        elif scheme == "categorical":
            cmap = _plt.get_cmap(params["name"])
            uniq = list(dict.fromkeys(gdf[fld].dropna()))
            idx = {v: i for i, v in enumerate(uniq)}
            arr = np.array([_rgba255(cmap(idx.get(v, 0) % cmap.N)) for v in gdf[fld]], dtype=np.uint8)
        elif scheme == "constant_num":
            wid = np.full(n, float(params["value"]))
        elif scheme == "scalar":
            d, r = params["domain"], params["range"]
            vals = gdf[fld].to_numpy(dtype=float)
            t = np.clip((vals - d[0]) / max(d[1] - d[0], 1e-12), 0, 1)
            wid = r[0] + t * (r[1] - r[0])
        if arr is not None:
            if any("stroke" in e for e in encs):
                out["stroke"] = arr
            if any("fill" in e for e in encs):
                out["fill"] = arr
        if wid is not None and any("width" in e for e in encs):
            out["width"] = wid
    return out


# --- the map document -------------------------------------------------------
class MapDoc:
    """Collects styled layers; displays via lonboard (WebGL) or matplotlib."""

    def __init__(self):
        self.items = []  # (gdf, name, arrays, opacity)

    def add(self, gdf, name, symbology, opacity):
        g = gdf.reset_index(drop=True).explode(index_parts=False).reset_index(drop=True)
        self.items.append((g, name, _style_arrays(symbology, g), opacity))

    def _lonboard_map(self):
        from lonboard import Map, PathLayer, PolygonLayer, ScatterplotLayer
        layers = []
        for g, name, st, op in self.items:
            if not len(g):
                continue
            geom = g.geometry.geom_type.iloc[0]
            base = g[["geometry"]]
            if "LineString" in geom:
                kw = {"width_units": "pixels", "width_min_pixels": 1.0, "opacity": op}
                if st["stroke"] is not None:
                    kw["get_color"] = st["stroke"]
                if st["width"] is not None:
                    kw["get_width"] = st["width"]
                layers.append(PathLayer.from_geopandas(base, **kw))
            elif "Polygon" in geom:
                kw = {"opacity": op * 0.6, "stroked": False}
                if st["fill"] is not None:
                    kw["get_fill_color"] = st["fill"]
                layers.append(PolygonLayer.from_geopandas(base, **kw))
            else:
                kw = {"radius_min_pixels": 5, "opacity": op}
                fill = st["fill"] if st["fill"] is not None else st["stroke"]
                if fill is not None:
                    kw["get_fill_color"] = fill
                layers.append(ScatterplotLayer.from_geopandas(base, **kw))
        return Map(layers=layers, basemap=None)

    def _static_figure(self):
        fig, ax = _plt.subplots(figsize=(9, 7))
        ax.set_facecolor("#eef1f4")
        for g, name, st, op in self.items:
            if not len(g):
                continue
            geom = g.geometry.geom_type.iloc[0]
            if "LineString" in geom:
                colors = st["stroke"] / 255 if st["stroke"] is not None else "#1d4ed8"
                widths = st["width"] if st["width"] is not None else 1.0
                g.plot(ax=ax, color=colors, linewidth=widths, alpha=op)
            elif "Polygon" in geom:
                colors = st["fill"] / 255 if st["fill"] is not None else "#cbd5e1"
                g.plot(ax=ax, color=colors, alpha=op * 0.6)
            else:
                fill = st["fill"] if st["fill"] is not None else st["stroke"]
                g.plot(ax=ax, color=(fill / 255 if fill is not None else "#dc2626"),
                       markersize=25, alpha=op)
        ax.set_aspect(1.4)
        ax.set_xticks([]), ax.set_yticks([])
        _plt.tight_layout()
        _plt.close(fig)
        return fig

    def _ipython_display_(self):
        from IPython.display import display
        be = os.environ.get("AEQ_MAP_BACKEND", "lonboard").strip().lower()
        display(self._static_figure() if be == "static" else self._lonboard_map())


def new_map(gdf_for_extent=None, zoom=12):
    """Create a map document (extent/zoom args kept for API compatibility;
    lonboard auto-fits to its layers)."""
    return MapDoc()


def add_gdf(doc, gdf, name, symbology=None, **kwargs):
    """Add a GeoDataFrame to the map as a styled layer."""
    doc.add(gdf, name, symbology, kwargs.get("opacity", 1.0))
    return name


def merge_lines(gdf, tol=0.01):
    """Collapse many lines into a single MultiLineString feature — backdrop
    layers do not need per-feature identity, and one merged feature is a
    fraction of the size and draw cost."""
    import geopandas as _gpd
    from shapely.geometry import MultiLineString
    parts = []
    for geom in gdf.geometry.simplify(tol):
        if geom is None or geom.is_empty:
            continue
        parts.extend(geom.geoms if geom.geom_type == "MultiLineString" else [geom])
    return _gpd.GeoDataFrame({"links": [len(parts)]}, geometry=[MultiLineString(parts)], crs=gdf.crs)


In [6]:
# field()/constant() symbology builders come from the map helper cell

doc = new_map(links, zoom=12)

add_gdf(doc, zones, "zones", opacity=0.4, symbology=[[constant("#f59e0b").encoding("fill")]])
add_gdf(doc, links, "links", symbology=[[constant("#1d4ed8").encoding("stroke")]])
add_gdf(doc, nodes[nodes.is_centroid == 1], "centroids", symbology=[[constant("#dc2626").encoding("fill")]])

doc  # render the map (interactive in JupyterLab)

[interactive offline map - run the notebook to display]

Try the layer tree: toggle layers, right-click one to open the *Symbology* editor,
or use `doc.export_to_qgis("model.qgz")` to hand the exact same map to QGIS.

## Closing up

Always close a project when done — it releases the SQLite connections and flushes logs.


In [7]:
project.close()

---
**Next:** [02 — Zones and centroid connectors](02_zones_and_connectors.ipynb), where we
build a zoning system from scratch and hook it to the network.
